# Eesti II samba fondide aktiivse juhtimise osakaal

Arvutame **aktiivse juhtimise osakaalu** (Active Share, Cremers & Petajisto 2009) Eesti II samba pensionifondidele. Kasutame andmetena Tuleva fondide röntgeni andmestikku mai-juuni 2026 seisuga. Andmestik on kättesaadav: https://github.com/StenEhrlich/estonian-pension-lookthrough

Iga fondi võrdlusindeksina kasutame sama fondivalitseja indeksfondist ning Tuleva Maailma Võlakirjade pensionifondi portfellidest loodud võrdlusindeksit.


In [1]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/StenEhrlich/estonian-pension-lookthrough/main"

fondid       = pd.read_csv(f"{BASE}/funds.csv", index_col="code")
varaklassid  = pd.read_csv(f"{BASE}/fund_sleeves.csv").pivot(index="code", columns="sleeve", values="weight_pct")
positsioonid = pd.read_csv(f"{BASE}/holdings.csv")   # kõik portfellid: aktiivsed fondid, indeksfondid, TUK00


Kõik portfellid on nüüd **ühes failis** `holdings.csv` (fondi roll — aktiivne fond või võrdlusindeks — on kirjas `funds.csv`-s). Failinimed ja veerunimed jäävad inglise keelde, sest andmestik on avalik ja rahvusvaheliselt kasutatav; notebook'i muutujad on töö terminites.

Kiire pilk andmetele:


In [2]:
print(f"{len(positsioonid):,} positsioonirida, {len(fondid)} fondi")
print(positsioonid.groupby("vector").size().to_string())
fondid[["name", "manager", "role", "benchmark_code"]]


107,831 positsioonirida, 23 fondi
vector
bond              11894
bond_ceiling         61
bond_floor         2782
bond_namekeyed     1197
equity            54396
equity_strict     35470
unit               2031


,name,manager,role,benchmark_code
code,,,,
K1960,Swedbank Pensionifond K1960,Swedbank,active,Ki
K1970,Swedbank Pensionifond K1970,Swedbank,active,Ki
K1980,Swedbank Pensionifond K1980,Swedbank,active,Ki
K1990,Swedbank Pensionifond K1990,Swedbank,active,Ki
K2000,Swedbank Pensionifond K2000,Swedbank,active,Ki
KKONS,Swedbank Pensionifond Konservatiivne,Swedbank,active,Ki
Ki,Swedbank Pensionifond Indeks,Swedbank,benchmark,Ki
LIK75,LHV Pensionifond Indeks,LHV,benchmark,LIK75
LLK50,LHV Pensionifond Ettevõtlik,LHV,active,LIK75


## 1. Valem

$$\text{Active Share} = \tfrac{1}{2}\sum_i \lvert w_{fond,i} - w_{indeks,i}\rvert$$

 0% tähendaks, et portfellid on identsed. 100% tähendab, et ühisosa puudub. Erinevuste summa jagatakse läbi kahega, et vältida topeltloendamist. Kui mõnda aktsiat on portfellis rohkem kui võrdlusindeksis, siis järelikult peab sama palju olema vähem mõnda teist aktsiat. 


In [3]:
def vek(kood, vektor):
    """Ühe fondi positsioonid: kaal % väärtpaberi/emitendi kaupa (summa = 100)."""
    read = positsioonid[(positsioonid.code == kood) & (positsioonid.vector == vektor)]
    return read.set_index("key").weight_pct

def AS(fond, indeks):
    return 0.5 * fond.sub(indeks, fill_value=0).abs().sum()

# käsitsi kontroll: 60/40 vs 50/50 -> 0.5*(10+10) = 10
assert AS(pd.Series({"A": 60, "B": 40}), pd.Series({"A": 50, "B": 50})) == 10


## 2. Kolm mõõdikut

1. **Aktsiaosa AS** — fondi aktsiad (läbi vaadatud) vs fondivalitseja indeksfondi aktsiad.
2. **Võlakirjaosa AS** — fondi võlakirjad **emitendi** tasandil (valitsused riigi, ettevõtted normaliseeritud nime järgi) vs Tuleva Võlakirjad (TUK00).
3. **Komposiit-AS** — kogu fond vs fondi **enda varajaotusega** kaalutud võrdlusindeks: aktsiaosa → indeksfond; võlakirjad + erakapital + kinnisvara + kuld → TUK00; raha neutraalne (võrdlusindeks „hoiab" sama palju raha kui fond). Erakapitalil, kinnisvaral ja kullal pole võrdlusindeksis vastet, seega loevad nad täies mahus aktiivseks.

**Kas aktsiaosa ja võlakirjaosa eraldi arvutamine on vajalik?** Jah — need on töö kolm *eraldi* mõõdikut. Aktsiaosa AS mõõdab puhast väärtpaberivalikut aktsiate sees, võlakirjaosa AS sama võlakirjades; komposiit mõõdab kogu portfelli. Komposiiti ei saa kahest esimesest kokku liita, sest selles on ka PE/kinnisvara/kulla panus.

**NB — metoodikarevisjonid 2026-07-19:** komposiitreegel on nüüd kõigil neljal fondivalitsejal ühesugune (varem oli SEB-il spetsist tulenev erand). SEB komposiidid tõusid kuni +2,7 pp; `reference_results.json` kannab uusi väärtusi (vt `METHOD_REVISIONS.md` analüüsirepos).

Reegel on üks, seega ei vaja me eraldi funktsioone — arvutame kõik jooksvalt ühes tsüklis. Kõigepealt aga sama arvutus samm-sammult ühe fondi peal.


## 3. Võrdlusindeksi määramine

Võrdlusindeksi valik on **metoodiline otsus ja tehakse siin, mitte andmestikus** — andmestik sisaldab ainult andmeid (portfellid, varajaotused, allikad). Iga fondi aktsiaosa võrdlusindeks on sama fondivalitseja enda indeksfond; võlakirjaosa võrdlusindeks on Tuleva Võlakirjad (TUK00). Kes teist võrdlust soovib (nt kõik fondid sama indeksi vastu), muudab ainult seda lahtrit.


In [4]:
indeksfond = {"SEB": "SIK75", "LHV": "LIK75", "Luminor": "NIK100", "Swedbank": "Ki"}


## 4. Näide: LHV Julge (LXK75)


In [5]:
kood = "LXK75"                                             # LHV Julge
kaalud = varaklassid.loc[kood] * 100 / varaklassid.loc[kood, "total"]
print("varaklasside kaalud (%):"); print(kaalud.drop("total").round(1).to_string())

aktsiad = vek(kood, "equity")                              # fondi aktsiad, summa 100
indeks  = vek(indeksfond[fondid.loc[kood, "manager"]], "equity")  # LHV Indeks läbi vaadatuna
print(f"\naktsiaosa AS = {AS(aktsiad, indeks):.1f}%  ({len(aktsiad)} aktsiat vs {len(indeks)} indeksis)")
aktsiad.sort_values(ascending=False).head()                # fondi suurimad aktsiapositsioonid


varaklasside kaalud (%):
sleeve
bond    16.5
cash     6.3
eq      46.6
gold     5.6
pe      16.7
re       8.3

aktsiaosa AS = 94.9%  (224 aktsiat vs 3942 indeksis)


key
N:FORTUM           8.438638
N:TOTALENERGIES    4.005140
N:STORA ENSO       3.019919
N:INVESTOR         2.869994
N:GLENCORE         2.727345
Name: weight_pct, dtype: float64

Komposiidi jaoks paneme kogu fondi ühte vektorisse: iga varaklassi vektor × tema kaal. Võrdlusindeks ehitatakse **samamoodi**, fondi enda kaaludega. Prefiks (`EQ:` / `BD:`) hoiab aktsiad ja võlakirjaemitendid lahus; PE, RE, KULD ja RAHA on üherealised „ämbrid".


In [ ]:
vlk   = vek(kood, "bond")
tuk00 = vek("TUK00", "bond")

fondi_jaotus = pd.concat([
    (aktsiad * kaalud["eq"] / 100).add_prefix("EQ:"),
    (vlk * kaalud["bond"] / 100).add_prefix("BD:"),
    pd.Series({"PE": kaalud["pe"], "RE": kaalud["re"], "KULD": kaalud["gold"], "RAHA": kaalud["cash"]})])

vordlusjaotus = pd.concat([
    (indeks * kaalud["eq"] / 100).add_prefix("EQ:"),
    (tuk00 * (kaalud["bond"] + kaalud["pe"] + kaalud["re"] + kaalud["gold"]) / 100).add_prefix("BD:"),
    pd.Series({"RAHA": kaalud["cash"]})])                  # raha neutraalne

panus = 0.5 * fondi_jaotus.sub(vordlusjaotus, fill_value=0).abs()
panus = panus.groupby(panus.index.str.split(":").str[0]).sum()
print(f"komposiit-AS = {panus.sum():.1f}%, sellest:"); print(panus.round(1).to_string())


## 5. Arvutus — kõik fondid

Sama arvutus tsüklis üle kõigi aktiivsete fondide.


In [ ]:
tulemuste_read = []
for kood, fond in fondid[fondid.role == "active"].iterrows():
    kaalud = varaklassid.loc[kood] * 100 / varaklassid.loc[kood, "total"]
    aktsiad, vlk = vek(kood, "equity"), vek(kood, "bond")
    indeks, tuk00 = vek(indeksfond[fond.manager], "equity"), vek("TUK00", "bond")

    fondi_jaotus = pd.concat([
        (aktsiad * kaalud["eq"] / 100).add_prefix("EQ:"),
        (vlk * kaalud["bond"] / 100).add_prefix("BD:"),
        pd.Series({"PE": kaalud["pe"], "RE": kaalud["re"], "KULD": kaalud["gold"], "RAHA": kaalud["cash"]})])
    vordlusjaotus = pd.concat([
        (indeks * kaalud["eq"] / 100).add_prefix("EQ:"),
        (tuk00 * (kaalud["bond"] + kaalud["pe"] + kaalud["re"] + kaalud["gold"]) / 100).add_prefix("BD:"),
        pd.Series({"RAHA": kaalud["cash"]})])

    panus = 0.5 * fondi_jaotus.sub(vordlusjaotus, fill_value=0).abs()
    panus = panus.groupby(panus.index.str.split(":").str[0]).sum()

    tulemuste_read.append({"kood": kood, "fond": fond["name"],
        "aktsia_AS": AS(aktsiad, indeks) if len(aktsiad) else None,
        "vlk_AS": AS(vlk, tuk00) if len(vlk) else None,
        "komposiit_AS": panus.sum(), **panus.add_prefix("pp_")})

tulemused = pd.DataFrame(tulemuste_read).set_index("kood").round(2)
tulemused[["fond", "aktsia_AS", "vlk_AS", "komposiit_AS"]]


## 6. Kontroll avaldatud tulemuste vastu

See plokk ei arvuta midagi uut — see on **kvaliteedikontroll andmestiku kasutajale**: kinnitus, et sinu masinas annab see kood täpselt samad numbrid, mis töös avaldatud (`reference_results.json`; uueneb koos metoodikaga). Kui andmestik ja kood lähevad omavahel vastuollu, kukub see lahter veaga läbi selle asemel, et vaikselt valesid numbreid näidata.


In [8]:
ref = pd.read_json(f"{BASE}/reference_results.json").T
for meie, avaldatud in [("aktsia_AS", "equity_AS"), ("vlk_AS", "bond_AS"), ("komposiit_AS", "composite_AS")]:
    erinevus = (tulemused[meie] - ref[avaldatud].astype(float)).abs()
    assert erinevus.max() < 0.01, (meie, erinevus.idxmax())
print(f"kõik {len(tulemused)} fondi klapivad avaldatud tulemustega")


kõik 18 fondi klapivad avaldatud tulemustega


## 7. Väljund töö jaoks

Arvutus toimus eespool — siin ainult vormistus: fondid töö tabeli järjekorda, LaTeX-read (`tab:esma` AS-veerg) ja CSV.


In [9]:
jrk = ["SEK100", "SEK50", "SEK25", "SEK00", "LXK75", "LLK50", "LMK25", "LXK00",
       "NPK75", "NPK50", "NPK25", "NPK00", "K1960", "K1970", "K1980", "K1990", "K2000", "KKONS"]
tabel = tulemused.loc[jrk]
tabel.to_csv("active_share_results.csv")
for kood, r in tabel.iterrows():
    print(f"{r['fond']:35s} & {r['komposiit_AS']:.1f} \\\\".replace(".", ","))
tabel


SEB Pensionifond 18+                & 40,5 \\
SEB Pensionifond 55+                & 44,5 \\
SEB Pensionifond 60+                & 55,8 \\
SEB Pensionifond 65+                & 55,9 \\
LHV Pensionifond Julge              & 84,2 \\
LHV Pensionifond Ettevõtlik         & 90,0 \\
LHV Pensionifond Tasakaalukas       & 80,5 \\
LHV Pensionifond Rahulik            & 76,9 \\
Luminor 16-50                       & 18,4 \\
Luminor 50-56                       & 29,7 \\
Luminor 56+                         & 49,0 \\
Luminor 61-65                       & 65,0 \\
Swedbank Pensionifond K1960         & 68,5 \\
Swedbank Pensionifond K1970         & 53,0 \\
Swedbank Pensionifond K1980         & 51,6 \\
Swedbank Pensionifond K1990         & 1,3 \\
Swedbank Pensionifond K2000         & 50,0 \\
Swedbank Pensionifond Konservatiivne & 76,3 \\


,fond,aktsia_AS,vlk_AS,komposiit_AS,pp_BD,pp_EQ,pp_KULD,pp_PE,pp_RAHA,pp_RE
kood,,,,,,,,,,
SEK100,SEB Pensionifond 18+,34.46,99.99,40.50,6.26,31.28,0.00,1.21,0.0,1.74
SEK50,SEB Pensionifond 55+,34.45,81.50,44.49,11.34,28.02,0.00,0.96,0.0,4.17
SEK25,SEB Pensionifond 60+,18.39,89.77,55.76,41.62,8.94,0.00,0.23,0.0,4.97
SEK00,SEB Pensionifond 65+,27.94,58.59,55.87,53.39,2.48,0.00,0.00,0.0,0.00
LXK75,LHV Pensionifond Julge,94.88,80.27,84.18,24.69,44.17,2.80,8.35,0.0,4.17
LLK50,LHV Pensionifond Ettevõtlik,95.37,80.27,89.96,33.00,30.08,2.87,16.81,0.0,7.20
LMK25,LHV Pensionifond Tasakaalukas,99.36,80.27,80.53,56.37,8.19,3.21,3.56,0.0,9.20
LXK00,LHV Pensionifond Rahulik,100.00,80.27,76.87,73.55,0.49,2.82,0.00,0.0,0.00
NPK75,Luminor 16-50,14.25,98.18,18.39,4.39,13.44,0.00,0.34,0.0,0.22
